# Viettel AI Race 2026 — Colab T4 validation

This notebook validates a clean vLLM installation, server stability, the repository's 420-request workload, and accuracy tooling. It is **not** an H200 performance benchmark: never choose a portal submission from T4 TTFT, TPOT, or ERS.

The T4 profile uses FP16 because a Tesla T4 has no native Hopper FP8 W8A8 path. The exact v6 FP8/FP8-KV flags remain available as an opt-in **startup smoke** profile; record whether it starts, but do not compare its latency with H200. `speculative-draft-v6-fp8-smoke` applies those exact target flags with the local draft, while `speculative-draft-smoke` stays FP16 for optional functional triage. Neither profile measures H200 latency.

If an earlier cell imported `torch` or `vllm`, choose **Runtime → Restart session** before running this notebook from the top.

## 1. Clone the repository and install the CUDA 12.9 vLLM wheel

The setup deliberately removes the incompatible PyPI/CUDA-13 installation and any legacy `libcudart.so.13` symlink. It then installs **only** `vllm==0.22.1` through the official `cu129` wheel index and verifies the compiled extension in a fresh subprocess before any model is downloaded.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = 'https://github.com/Platypus27-coder/viettel-ai-race-llm-serving.git'
REPO_DIR = Path('/content/viettel-ai-race-llm-serving')
ARTIFACT_ROOT = Path('/content/viettel-artifacts')
GREEDY_BASELINE_ROOT = Path('/content/viettel-greedy-baselines')
RUN_DIR = ARTIFACT_ROOT / f"colab-{time.strftime('%Y%m%d-%H%M%S')}"
MODEL_ID = 'LiquidAI/LFM2.5-1.2B-Instruct'
MODEL_REVISION = os.environ.get('VIETTEL_MODEL_REVISION', 'main')
MODEL_DIR = Path('/content/LFM2.5-1.2B-Instruct')
DRAFT_MODEL_ID = 'LiquidAI/LFM2.5-350M'
# This revision is intentionally immutable: it must match the draft baked into a portal image.
DRAFT_MODEL_REVISION = '1575d1b8b67d862834836087765bff2ef4020672'
DRAFT_MODEL_DIR = Path('/content/LFM2.5-350M')
BASE_URL = 'http://127.0.0.1:8000'
VALID_COLAB_PROFILES = {
    't4-fp16', 'v6-fp8-smoke', 'shortconv-fp8-smoke',
    'speculative-draft-smoke', 'speculative-draft-v6-fp8-smoke',
}
SPECULATIVE_DRAFT_PROFILES = {
    'speculative-draft-smoke', 'speculative-draft-v6-fp8-smoke',
}
SPECULATIVE_PORTAL_CANDIDATES = {
    'speculative-draft',
    'speculative-draft-batch1536',
    'speculative-draft-batch1024',
}
SPECULATIVE_PORTAL_SCHEDULER_ARGS = {
    'speculative-draft': (),
    'speculative-draft-batch1536': ('--max-num-batched-tokens=1536',),
    'speculative-draft-batch1024': ('--max-num-batched-tokens=1024',),
}
# Select the profile before setup: it determines the immutable Git checkout and image evidence.
COLAB_PROFILE = os.environ.get('VIETTEL_COLAB_PROFILE', 't4-fp16').strip()
if COLAB_PROFILE not in VALID_COLAB_PROFILES:
    raise ValueError('VIETTEL_COLAB_PROFILE must be one of: ' + ', '.join(sorted(VALID_COLAB_PROFILES)))
PROFILE_AT_SETUP = COLAB_PROFILE

# A speculative preflight must never silently exercise main. An explicit immutable
# commit is accepted; otherwise the guarded candidate branch is the safe default.
DEFAULT_REPO_REF = 'candidate/speculative-draft' if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES else 'main'
REPO_REF = os.environ.get('VIETTEL_REPO_REF', DEFAULT_REPO_REF).strip()
EXPECTED_REPO_SHA = os.environ.get('VIETTEL_EXPECTED_REPO_SHA', '').strip().lower() or None
if EXPECTED_REPO_SHA and not re.fullmatch(r'[0-9a-f]{40}', EXPECTED_REPO_SHA):
    raise ValueError('VIETTEL_EXPECTED_REPO_SHA must be a 40-character lowercase commit SHA')
MAIN_REFS = {'main', 'origin/main', 'refs/heads/main', 'refs/remotes/origin/main'}
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES and REPO_REF in MAIN_REFS:
    raise ValueError(
        'Speculative profiles require VIETTEL_REPO_REF to be a non-main candidate ref or commit; '
        'omit it to use candidate/speculative-draft.'
    )
PORTAL_CANDIDATE_NAME = os.environ.get('VIETTEL_PORTAL_CANDIDATE', 'speculative-draft').strip()
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES and PORTAL_CANDIDATE_NAME not in SPECULATIVE_PORTAL_CANDIDATES:
    raise ValueError(
        'VIETTEL_PORTAL_CANDIDATE must be one of: ' + ', '.join(sorted(SPECULATIVE_PORTAL_CANDIDATES))
    )

# Portal Compose must point to a public Docker Hub image pinned by digest. The
# selector repeats this validation when rendering the archived Compose.
DOCKER_HUB_DIGEST_IMAGE = re.compile(
    r'^(?:(?:docker\.io|index\.docker\.io)/)?'
    r'[a-z0-9](?:[a-z0-9_-]*[a-z0-9])?/'
    r'[a-z0-9](?:[a-z0-9._-]*[a-z0-9])?'
    r'(?::[A-Za-z0-9_][A-Za-z0-9_.-]*)?@sha256:[0-9a-fA-F]{64}$'
)
VIETTEL_IMAGE_REFERENCE = os.environ.get('VIETTEL_IMAGE_REFERENCE', '').strip()
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES and not DOCKER_HUB_DIGEST_IMAGE.fullmatch(VIETTEL_IMAGE_REFERENCE):
    raise ValueError(
        'Speculative profiles require VIETTEL_IMAGE_REFERENCE as a public Docker Hub '
        'namespace/repository@sha256:<64-hex-digest>.'
    )

def run_checked(command: list[str], **kwargs) -> subprocess.CompletedProcess[str]:
    print('$', ' '.join(command))
    kwargs.setdefault('check', True)
    kwargs.setdefault('text', True)
    return subprocess.run(command, **kwargs)

loaded_runtime_modules = sorted({'torch', 'vllm'} & set(sys.modules))
if loaded_runtime_modules:
    raise RuntimeError(
        'This kernel already imported ' + ', '.join(loaded_runtime_modules)
        + '. Select Runtime → Restart session, then run this notebook from the top.'
    )

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
GREEDY_BASELINE_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)

# A clone/fetch/checkout sequence works on a fresh runtime and can also refresh
# a retained /content directory without relying on an uploaded project folder.
if not REPO_DIR.exists():
    run_checked(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} exists but is not the expected Git clone; remove it and rerun.')

run_checked(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL])
run_checked(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_REF])
run_checked(['git', '-C', str(REPO_DIR), 'checkout', '--detach', '--force', 'FETCH_HEAD'])
REPO_SHA = run_checked(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True).stdout.strip()
if EXPECTED_REPO_SHA and REPO_SHA != EXPECTED_REPO_SHA:
    raise RuntimeError(
        f'Checked out {REPO_SHA}, but VIETTEL_EXPECTED_REPO_SHA requires {EXPECTED_REPO_SHA}. '
        'Restart from setup after correcting the immutable reference.'
    )
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES:
    print(f'SPECULATIVE PRE-FLIGHT locked to repository commit {REPO_SHA} from {REPO_REF!r}.')

required_repo_files = [
    REPO_DIR / 'benchmark' / 'benchmark_ers.py',
    REPO_DIR / 'benchmark' / 'compare_greedy.py',
    REPO_DIR / 'benchmark' / 'test_accuracy.py',
    REPO_DIR / 'scripts' / 'select_submission.py',
    REPO_DIR / 'docker' / 'shortconv-fp8' / 'patch_vllm_shortconv_fp8.py',
    REPO_DIR / '019e649f-4e27-74db-82da-920f57b13786' / 'grading-workload-spec.json',
    REPO_DIR / 'docker-compose.yml',
]
missing_repo_files = [str(path) for path in required_repo_files if not path.is_file()]
if missing_repo_files:
    raise RuntimeError('Repository checkout is incomplete: ' + ', '.join(missing_repo_files))

V6_SOURCE_COMPOSE = REPO_DIR / 'docker-compose.yml'
V6_SOURCE_COMPOSE_SHA256 = hashlib.sha256(V6_SOURCE_COMPOSE.read_bytes()).hexdigest()

# Delete only the broken symlink created by the old notebook. Never fabricate a
# CUDA-13 runtime from a CUDA-12 library.
legacy_cudart_link = Path('/usr/local/lib/libcudart.so.13')
removed_legacy_symlink = False
if legacy_cudart_link.is_symlink():
    legacy_cudart_link.unlink()
    removed_legacy_symlink = True

run_checked([sys.executable, '-m', 'pip', 'uninstall', '-y', 'vllm', 'torch', 'torchvision', 'torchaudio'], check=False)
run_checked([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)
run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'])

VLLM_WHEEL_INDEX = 'https://wheels.vllm.ai/0.22.1/cu129'
install_command = [
    sys.executable, '-m', 'uv', 'pip', 'install', '--system',
    '--torch-backend=cu129',
    '--extra-index-url', VLLM_WHEEL_INDEX,
    '--index-strategy', 'unsafe-best-match',
    'vllm==0.22.1',
    'aiohttp>=3.9.0', 'openai>=1.0.0', 'numpy>=1.24.0', 'transformers>=4.57.2',
]
run_checked(install_command)

preflight_code = r'''
import json
import torch
import vllm
import vllm._C
payload = {
    'vllm_version': vllm.__version__,
    'compiled_extension_imported': True,
    'torch_version': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_capability': torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
}
print('PREFLIGHT_JSON=' + json.dumps(payload, sort_keys=True))
'''
preflight = subprocess.run(
    [sys.executable, '-c', preflight_code], capture_output=True, text=True, check=False
)
(RUN_DIR / 'preflight.stdout.log').write_text(preflight.stdout, encoding='utf-8')
(RUN_DIR / 'preflight.stderr.log').write_text(preflight.stderr, encoding='utf-8')
marker_lines = [line for line in preflight.stdout.splitlines() if line.startswith('PREFLIGHT_JSON=')]
if preflight.returncode != 0 or not marker_lines:
    raise RuntimeError(
        'vLLM CUDA preflight failed before model download.\n'
        + (preflight.stdout + '\n' + preflight.stderr)[-5000:]
    )
environment = json.loads(marker_lines[-1].split('=', 1)[1])
preflight_errors = []
if environment['vllm_version'] != '0.22.1':
    preflight_errors.append(f"Expected vLLM 0.22.1, got {environment['vllm_version']}")
if not environment['compiled_extension_imported']:
    preflight_errors.append('vllm._C did not import')
if not str(environment['torch_cuda']).startswith('12.'):
    preflight_errors.append(f"Expected a CUDA 12 build, got {environment['torch_cuda']}")
if not environment['cuda_available'] or 'T4' not in str(environment['gpu_name']):
    preflight_errors.append(f"Expected a Tesla T4 GPU, got {environment['gpu_name']}")
if preflight_errors:
    raise RuntimeError('Preflight rejected this runtime: ' + '; '.join(preflight_errors))

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
(RUN_DIR / 'nvidia-smi.txt').write_text(smi.stdout + smi.stderr, encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(RUN_DIR / 'pip-freeze.txt').write_text(freeze.stdout, encoding='utf-8')
environment.update({
    'repository_url': REPO_URL,
    'repository_ref_requested': REPO_REF,
    'repository_commit': REPO_SHA,
    'repository_commit_expected': EXPECTED_REPO_SHA,
    'colab_profile': COLAB_PROFILE,
    'portal_image_reference': VIETTEL_IMAGE_REFERENCE or None,
    'source_compose_sha256': V6_SOURCE_COMPOSE_SHA256,
    'model_id': MODEL_ID,
    'model_revision_requested': MODEL_REVISION,
    'vllm_wheel_index': VLLM_WHEEL_INDEX,
    'removed_legacy_libcudart_symlink': removed_legacy_symlink,
})
(RUN_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))
print(f'Artifacts will be written to: {RUN_DIR}')


## 2. Download the target and required pinned draft after the environment preflight passes

This cell is intentionally after the compiled-extension check, so a CUDA mismatch never wastes time downloading model weights. Select the profile, candidate Git ref, and digest-pinned image **before setup**: a speculative profile defaults to `candidate/speculative-draft` and rejects `main`. It also downloads the immutable LFM2.5-350M draft locally and rejects the run unless its complete tokenizer vocabulary and special-token configuration exactly match the target. Restart the session before changing profiles.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer

# The setup checkout, profile, and image evidence are one immutable preflight identity.
COLAB_PROFILE = os.environ.get('VIETTEL_COLAB_PROFILE', COLAB_PROFILE).strip()
if COLAB_PROFILE not in VALID_COLAB_PROFILES:
    raise ValueError('VIETTEL_COLAB_PROFILE must be one of: ' + ', '.join(sorted(VALID_COLAB_PROFILES)))
if COLAB_PROFILE != PROFILE_AT_SETUP:
    raise RuntimeError(
        'VIETTEL_COLAB_PROFILE changed after setup. Restart the Colab session and rerun '
        'from setup so the repository ref and image evidence remain reproducible.'
    )
if os.environ.get('VIETTEL_REPO_REF', DEFAULT_REPO_REF).strip() != REPO_REF:
    raise RuntimeError('VIETTEL_REPO_REF changed after setup; restart and rerun from setup.')
if (os.environ.get('VIETTEL_EXPECTED_REPO_SHA', '').strip().lower() or None) != EXPECTED_REPO_SHA:
    raise RuntimeError('VIETTEL_EXPECTED_REPO_SHA changed after setup; restart and rerun from setup.')
if os.environ.get('VIETTEL_IMAGE_REFERENCE', '').strip() != VIETTEL_IMAGE_REFERENCE:
    raise RuntimeError('VIETTEL_IMAGE_REFERENCE changed after setup; restart and rerun from setup.')
if os.environ.get('VIETTEL_PORTAL_CANDIDATE', 'speculative-draft').strip() != PORTAL_CANDIDATE_NAME:
    raise RuntimeError('VIETTEL_PORTAL_CANDIDATE changed after setup; restart and rerun from setup.')
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES and not DOCKER_HUB_DIGEST_IMAGE.fullmatch(VIETTEL_IMAGE_REFERENCE):
    raise RuntimeError('Speculative preflight lost its valid Docker Hub digest image reference.')
PROFILE_AT_DOWNLOAD = COLAB_PROFILE

model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
MODEL_RESOLVED_REVISION = model_info.sha
print(f'Downloading {MODEL_ID}@{MODEL_RESOLVED_REVISION} to {MODEL_DIR}')
snapshot_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_RESOLVED_REVISION,
    local_dir=str(MODEL_DIR),
)
if not (MODEL_DIR / 'config.json').is_file():
    raise RuntimeError(f'Model download did not create config.json under {MODEL_DIR}')
model_manifest = {
    'model_id': MODEL_ID,
    'revision_requested': MODEL_REVISION,
    'revision_resolved': MODEL_RESOLVED_REVISION,
    'local_path': str(MODEL_DIR),
    'snapshot_path': str(snapshot_path),
}

DRAFT_MODEL_RESOLVED_REVISION = None
tokenizer_validation = None
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES:
    draft_info = HfApi().model_info(DRAFT_MODEL_ID, revision=DRAFT_MODEL_REVISION)
    DRAFT_MODEL_RESOLVED_REVISION = draft_info.sha
    if DRAFT_MODEL_RESOLVED_REVISION != DRAFT_MODEL_REVISION:
        raise RuntimeError(
            'Draft revision was not immutable: expected ' + DRAFT_MODEL_REVISION
            + ', got ' + str(DRAFT_MODEL_RESOLVED_REVISION)
        )
    print(f'Downloading pinned draft {DRAFT_MODEL_ID}@{DRAFT_MODEL_RESOLVED_REVISION} to {DRAFT_MODEL_DIR}')
    draft_snapshot_path = snapshot_download(
        repo_id=DRAFT_MODEL_ID,
        revision=DRAFT_MODEL_RESOLVED_REVISION,
        local_dir=str(DRAFT_MODEL_DIR),
    )
    if not (DRAFT_MODEL_DIR / 'config.json').is_file():
        raise RuntimeError(f'Draft download did not create config.json under {DRAFT_MODEL_DIR}')

    def json_safe(value):
        if isinstance(value, dict):
            return {str(key): json_safe(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_safe(item) for item in value]
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return str(value)

    def tokenizer_summary(tokenizer):
        vocabulary = tokenizer.get_vocab()
        normalized_vocabulary = sorted((str(token), int(token_id)) for token, token_id in vocabulary.items())
        vocabulary_bytes = json.dumps(
            normalized_vocabulary, ensure_ascii=False, separators=(',', ':')
        ).encode('utf-8')
        return vocabulary, {
            'tokenizer_class': tokenizer.__class__.__name__,
            'vocab_size': len(tokenizer),
            'base_vocab_size': tokenizer.vocab_size,
            'vocab_sha256': hashlib.sha256(vocabulary_bytes).hexdigest(),
            'special_tokens_map': json_safe(tokenizer.special_tokens_map),
            'all_special_tokens': [str(token) for token in tokenizer.all_special_tokens],
            'all_special_ids': [int(token_id) for token_id in tokenizer.all_special_ids],
            'bos_token_id': tokenizer.bos_token_id,
            'eos_token_id': tokenizer.eos_token_id,
            'pad_token_id': tokenizer.pad_token_id,
            'unk_token_id': tokenizer.unk_token_id,
        }

    target_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
    draft_tokenizer = AutoTokenizer.from_pretrained(str(DRAFT_MODEL_DIR), trust_remote_code=True)
    target_vocabulary, target_tokenizer_summary = tokenizer_summary(target_tokenizer)
    draft_vocabulary, draft_tokenizer_summary = tokenizer_summary(draft_tokenizer)
    compatibility_errors = []
    if target_vocabulary != draft_vocabulary:
        compatibility_errors.append('complete tokenizer vocabulary/token-ID mapping differs')
    for field in ('special_tokens_map', 'all_special_tokens', 'all_special_ids', 'bos_token_id', 'eos_token_id', 'pad_token_id', 'unk_token_id'):
        if target_tokenizer_summary[field] != draft_tokenizer_summary[field]:
            compatibility_errors.append(f'special-token field differs: {field}')
    tokenizer_validation = {
        'target_model_id': MODEL_ID,
        'target_model_revision_resolved': MODEL_RESOLVED_REVISION,
        'draft_model_id': DRAFT_MODEL_ID,
        'draft_model_revision_pinned': DRAFT_MODEL_REVISION,
        'draft_model_revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
        'target': target_tokenizer_summary,
        'draft': draft_tokenizer_summary,
        'compatible': not compatibility_errors,
        'errors': compatibility_errors,
    }
    (RUN_DIR / 'tokenizer_compatibility.json').write_text(
        json.dumps(tokenizer_validation, indent=2), encoding='utf-8'
    )
    if compatibility_errors:
        raise RuntimeError(
            'Draft tokenizer is incompatible with target; inspect tokenizer_compatibility.json: '
            + '; '.join(compatibility_errors)
        )
    model_manifest['draft'] = {
        'model_id': DRAFT_MODEL_ID,
        'revision_pinned': DRAFT_MODEL_REVISION,
        'revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
        'local_path': str(DRAFT_MODEL_DIR),
        'snapshot_path': str(draft_snapshot_path),
        'tokenizer_compatible': True,
    }

model_manifest['profile'] = COLAB_PROFILE
(RUN_DIR / 'model_manifest.json').write_text(json.dumps(model_manifest, indent=2), encoding='utf-8')
environment.update({
    'colab_profile': COLAB_PROFILE,
    'model_revision_resolved': MODEL_RESOLVED_REVISION,
    'draft_model_id': DRAFT_MODEL_ID if DRAFT_MODEL_RESOLVED_REVISION else None,
    'draft_model_revision_pinned': DRAFT_MODEL_REVISION if DRAFT_MODEL_RESOLVED_REVISION else None,
    'draft_model_revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
})
(RUN_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(model_manifest, indent=2))


## 3. Start a clean server

`t4-fp16` is the default functional/stability profile. Run `t4-fp16` or `v6-fp8-smoke` first to capture its post-workload greedy parent artifact; the corresponding speculative profile compares against it after its own workload. Set the speculative profile, its non-main Git ref (default: `candidate/speculative-draft`), digest `VIETTEL_IMAGE_REFERENCE=namespace/repository@sha256:...`, and optional `VIETTEL_PORTAL_CANDIDATE` (`speculative-draft`, `speculative-draft-batch1536`, or `speculative-draft-batch1024`) **before the setup cell**. A scheduler child is rendered from a digest-pinned draft parent and adds only its declared batch-token flag to the Colab source-equivalent command. Restart the session before changing profiles.

In [ ]:
import shutil
import urllib.error
import urllib.request

requested_profile = os.environ.get('VIETTEL_COLAB_PROFILE', PROFILE_AT_DOWNLOAD).strip()
if requested_profile != PROFILE_AT_DOWNLOAD or COLAB_PROFILE != PROFILE_AT_DOWNLOAD:
    raise RuntimeError(
        'VIETTEL_COLAB_PROFILE changed after model download. Rerun the model-download cell '
        'so profile-dependent assets and validation are recorded.'
    )
if COLAB_PROFILE not in VALID_COLAB_PROFILES:
    raise ValueError('VIETTEL_COLAB_PROFILE changed after setup; restart from setup to select a valid profile.')
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES:
    if DRAFT_MODEL_RESOLVED_REVISION != DRAFT_MODEL_REVISION or tokenizer_validation is None:
        raise RuntimeError(
            'The speculative draft was not downloaded and validated. Set a speculative-draft '
            "profile, then rerun the model-download cell before starting the server."
        )
    if not tokenizer_validation.get('compatible'):
        raise RuntimeError('Draft tokenizer validation failed; inspect tokenizer_compatibility.json')

if 'server_process' in globals() and server_process.poll() is None:
    print('Stopping the prior vLLM server before starting a clean profile.')
    server_process.terminate()
    try:
        server_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server_process.kill()
if 'server_log_handle' in globals() and not server_log_handle.closed:
    server_log_handle.close()

ACTIVE_RUN_DIR = RUN_DIR / f"{COLAB_PROFILE}-{time.strftime('%Y%m%d-%H%M%S')}"
ACTIVE_RUN_DIR.mkdir(parents=True, exist_ok=False)
source_compose_artifact = ACTIVE_RUN_DIR / 'docker-compose.v6-reference.yml'
shutil.copy2(V6_SOURCE_COMPOSE, source_compose_artifact)

shortconv_patch_applied = COLAB_PROFILE == 'shortconv-fp8-smoke'
speculative_draft_enabled = COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES
speculative_portal_target = COLAB_PROFILE == 'speculative-draft-v6-fp8-smoke'
PORTAL_CANDIDATE = None
portal_scheduler_args = ()
if speculative_draft_enabled:
    if not DOCKER_HUB_DIGEST_IMAGE.fullmatch(VIETTEL_IMAGE_REFERENCE):
        raise RuntimeError('Speculative preflight requires a validated Docker Hub digest image reference.')
    portal_scheduler_args = SPECULATIVE_PORTAL_SCHEDULER_ARGS[PORTAL_CANDIDATE_NAME]
    render_log_path = ACTIVE_RUN_DIR / 'candidate-compose-render.log'

    def render_candidate(candidate: str, source: Path, output: Path, custom_image: str | None = None) -> None:
        command = [
            sys.executable, str(REPO_DIR / 'scripts' / 'select_submission.py'),
            '--candidate', candidate, '--source', str(source), '--output', str(output),
        ]
        if custom_image:
            command.extend(['--custom-image', custom_image])
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        with render_log_path.open('a', encoding='utf-8') as render_log:
            render_log.write('$ ' + ' '.join(command) + '\n')
            render_log.write(result.stdout + result.stderr)
        if result.returncode != 0 or not output.is_file():
            raise RuntimeError('Candidate Compose renderer failed; inspect candidate-compose-render.log')

    parent_compose_artifact = None
    candidate_compose_artifact = ACTIVE_RUN_DIR / f'docker-compose.{PORTAL_CANDIDATE_NAME}.yml'
    if PORTAL_CANDIDATE_NAME == 'speculative-draft':
        render_candidate(PORTAL_CANDIDATE_NAME, V6_SOURCE_COMPOSE, candidate_compose_artifact, VIETTEL_IMAGE_REFERENCE)
    else:
        parent_compose_artifact = ACTIVE_RUN_DIR / 'docker-compose.speculative-draft-parent.yml'
        render_candidate('speculative-draft', V6_SOURCE_COMPOSE, parent_compose_artifact, VIETTEL_IMAGE_REFERENCE)
        render_candidate(PORTAL_CANDIDATE_NAME, parent_compose_artifact, candidate_compose_artifact)
    candidate_compose_sha256 = hashlib.sha256(candidate_compose_artifact.read_bytes()).hexdigest()
    PORTAL_CANDIDATE = {
        'candidate': PORTAL_CANDIDATE_NAME,
        'image_reference': VIETTEL_IMAGE_REFERENCE,
        'image_digest': VIETTEL_IMAGE_REFERENCE.rsplit('@', 1)[1],
        'compose_sha256': candidate_compose_sha256,
        'compose_artifact': str(candidate_compose_artifact),
        'source_compose_sha256': V6_SOURCE_COMPOSE_SHA256,
        'source_compose_artifact': str(source_compose_artifact),
        'parent_compose_artifact': str(parent_compose_artifact) if parent_compose_artifact else None,
        'scheduler_arguments': list(portal_scheduler_args),
        'source_equivalent_preflight': speculative_portal_target,
        'preflight_mode': (
            'source-equivalent-not-container-image' if speculative_portal_target
            else 'functional-triage-not-source-equivalent'
        ),
        'repository_commit': REPO_SHA,
        'profile': COLAB_PROFILE,
    }
if shortconv_patch_applied:
    patch_command = [
        sys.executable,
        str(REPO_DIR / 'docker' / 'shortconv-fp8' / 'patch_vllm_shortconv_fp8.py'),
        '--apply', '--verify',
    ]
    patch_result = subprocess.run(patch_command, capture_output=True, text=True, check=False)
    (ACTIVE_RUN_DIR / 'shortconv_patch.log').write_text(
        patch_result.stdout + patch_result.stderr, encoding='utf-8'
    )
    if patch_result.returncode != 0:
        raise RuntimeError('ShortConv patch failed; inspect shortconv_patch.log')


v6_common_args = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    f'--model={MODEL_DIR}',
    '--served-model-name=LFM2.5-1.2B-Instruct',
    '--host=0.0.0.0', '--port=8000', '--tensor-parallel-size=1',
    '--max-model-len=8192',
    '--gpu-memory-utilization=0.97',
    '--enable-prefix-caching',
]
speculative_config = None
if COLAB_PROFILE in {'v6-fp8-smoke', 'shortconv-fp8-smoke'}:
    profile_args = ['--quantization=fp8', '--kv-cache-dtype=fp8_e4m3']
    profile_note = (
        'ShortConv FP8 patched startup/function/accuracy smoke only.'
        if shortconv_patch_applied else
        'Exact v6 FP8/FP8-KV startup smoke only.'
    ) + ' T4 is not an H200 FP8 latency proxy.'
elif speculative_draft_enabled:
    speculative_config = {
        'method': 'draft_model',
        'model': str(DRAFT_MODEL_DIR),
        'num_speculative_tokens': 4,
        'draft_tensor_parallel_size': 1,
        'max_model_len': 8192,
    }
    if speculative_portal_target:
        profile_args = [
            '--quantization=fp8', '--kv-cache-dtype=fp8_e4m3',
            '--speculative-config=' + json.dumps(speculative_config, separators=(',', ':')),
        ]
        profile_note = (
            'Exact v6 FP8/FP8-KV target plus pinned local draft startup/workload/accuracy smoke. '
            'T4 speculative latency is not an H200 performance signal.'
        )
    else:
        profile_args = [
            '--dtype=float16',
            '--speculative-config=' + json.dumps(speculative_config, separators=(',', ':')),
        ]
        profile_note = (
            'Pinned local draft FP16 startup/workload/accuracy smoke only. '
            'T4 speculative latency is not an H200 performance signal.'
        )
else:
    profile_args = ['--dtype=float16']
    profile_note = (
        'T4 functional, workload, and accuracy profile. Its latency must not select a portal candidate.'
    )

if speculative_draft_enabled and portal_scheduler_args:
    # This is the sole scheduler delta in the rendered child Compose as well.
    profile_args = [*profile_args, *portal_scheduler_args]
    profile_note += ' Source-equivalent scheduler child: ' + ', '.join(portal_scheduler_args) + '.'

vllm_cmd = [*v6_common_args, *profile_args]
server_env = os.environ.copy()
server_env.update({
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'VLLM_NO_USAGE_STATS': '1',
    'DO_NOT_TRACK': '1',
    # The server must not resolve/download any artifact after model staging.
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    # INFO is used in Colab only so resolved scheduler/Mamba settings are preserved.
    'VLLM_LOGGING_LEVEL': 'INFO',
})
offline_serving_env = {
    'HF_HUB_OFFLINE': server_env.get('HF_HUB_OFFLINE'),
    'TRANSFORMERS_OFFLINE': server_env.get('TRANSFORMERS_OFFLINE'),
}
if offline_serving_env != {'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}:
    raise RuntimeError('Offline serving is not enforced; refusing to start the source-equivalent server.')
recorded_server_env = {
    key: server_env[key] for key in (
        'PYTORCH_CUDA_ALLOC_CONF', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS',
        'VLLM_NO_USAGE_STATS', 'DO_NOT_TRACK', 'HF_HUB_OFFLINE',
        'TRANSFORMERS_OFFLINE', 'VLLM_LOGGING_LEVEL',
    )
}
source_equivalent_command_artifact = ACTIVE_RUN_DIR / 'source_equivalent_server_command.json'
source_equivalent_command = {
    'schema_version': 1,
    'captured_before_server_start': True,
    'repository_commit': REPO_SHA,
    'profile': COLAB_PROFILE,
    'source_equivalent_preflight': bool(
        PORTAL_CANDIDATE and PORTAL_CANDIDATE['source_equivalent_preflight']
    ),
    'portal_candidate': PORTAL_CANDIDATE,
    'command': vllm_cmd,
    'environment': recorded_server_env,
    'offline_serving': offline_serving_env,
    'target_model_local_path': str(MODEL_DIR),
    'draft_model_local_path': str(DRAFT_MODEL_DIR) if speculative_draft_enabled else None,
}
source_equivalent_command_artifact.write_text(
    json.dumps(source_equivalent_command, indent=2), encoding='utf-8'
)
server_config = {
    'profile': COLAB_PROFILE,
    'profile_selected_before_setup': PROFILE_AT_SETUP,
    'profile_selected_before_download': PROFILE_AT_DOWNLOAD,
    'profile_note': profile_note,
    'command': vllm_cmd,
    'common_v6_flags': {
        'max_model_len': 8192,
        'gpu_memory_utilization': 0.97,
        'prefix_caching': True,
    },
    'profile_flags': profile_args,
    'recorded_server_environment': recorded_server_env,
    'offline_serving': offline_serving_env,
    'source_equivalent_command_artifact': str(source_equivalent_command_artifact),
    'shortconv_patch_applied': shortconv_patch_applied,
    'speculative_draft_enabled': speculative_draft_enabled,
    'speculative_portal_target': speculative_portal_target,
    'speculative_config': speculative_config,
    'draft_model': ({
        'model_id': DRAFT_MODEL_ID,
        'revision_pinned': DRAFT_MODEL_REVISION,
        'revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
        'local_path': str(DRAFT_MODEL_DIR),
        'tokenizer_compatibility_artifact': str(RUN_DIR / 'tokenizer_compatibility.json'),
    } if speculative_draft_enabled else None),
    'repository_commit': REPO_SHA,
    'repository_ref_requested': REPO_REF,
    'model_revision_resolved': MODEL_RESOLVED_REVISION,
    'portal_candidate': PORTAL_CANDIDATE,
    'preflight_identity': {
        'source_equivalent_preflight': bool(
            PORTAL_CANDIDATE and PORTAL_CANDIDATE['source_equivalent_preflight']
        ),
        'source_compose_sha256': V6_SOURCE_COMPOSE_SHA256,
        'image_reference': VIETTEL_IMAGE_REFERENCE or None,
    },
}
(ACTIVE_RUN_DIR / 'server_config.json').write_text(json.dumps(server_config, indent=2), encoding='utf-8')

server_log_path = ACTIVE_RUN_DIR / 'vllm.log'
server_log_handle = server_log_path.open('w', encoding='utf-8', buffering=1)
print('Starting:', ' '.join(vllm_cmd))
print('WARNING: T4 latency is not an H200 performance signal.')
server_process = subprocess.Popen(
    vllm_cmd, stdout=server_log_handle, stderr=subprocess.STDOUT, env=server_env
)

resolved_startup_evidence = None

def capture_resolved_config() -> None:
    # vLLM does not expose every resolved engine setting through an API. Preserve the
    # raw evidence and make only detected fields machine-readable; absent fields stay
    # explicitly unresolved rather than being inferred from requested CLI flags.
    global resolved_startup_evidence
    server_log_handle.flush()
    log_text = server_log_path.read_text(encoding='utf-8', errors='replace')
    markers = (
        'engine args', 'scheduler', 'max_num_batched_tokens', 'max-num-batched-tokens',
        'max_num_seqs', 'max-num-seqs', 'chunked prefill', 'enable_chunked_prefill',
        'mamba', 'hybrid', 'mamba_cache_mode', 'prefix caching', 'enable_prefix_caching',
        'kv cache', 'kv_cache_dtype', 'max_model_len',
    )
    lines = log_text.splitlines()
    selected = [line for line in lines if any(marker in line.lower() for marker in markers)]
    evidence_mode = 'matched-log-lines'
    if not selected:
        selected = lines[-200:]
        evidence_mode = 'last-200-log-lines-fallback'
    resolved_log_path = ACTIVE_RUN_DIR / 'startup_resolved_config.log'
    resolved_log_path.write_text(
        '\n'.join(selected) + ('\n' if selected else ''), encoding='utf-8'
    )

    field_aliases = {
        'max_model_len': ('max_model_len', 'max-model-len'),
        'max_num_batched_tokens': ('max_num_batched_tokens', 'max-num-batched-tokens'),
        'max_num_seqs': ('max_num_seqs', 'max-num-seqs'),
        'chunked_prefill': ('chunked prefill', 'enable_chunked_prefill'),
        'mamba_cache_mode': ('mamba_cache_mode', 'mamba cache mode'),
        'hybrid_cache': ('hybrid cache', 'hybrid'),
        'prefix_caching': ('prefix caching', 'enable_prefix_caching'),
        'kv_cache_dtype': ('kv_cache_dtype', 'kv cache dtype'),
    }
    fields = {}
    for field, aliases in field_aliases.items():
        matches = [line for line in lines if any(alias in line.lower() for alias in aliases)]
        value = None
        if matches:
            value_match = re.search(r'(?:=|:)[ ]*([^, )]+)', matches[-1])
            if value_match:
                value = value_match.group(1).strip("'\"")
        fields[field] = {
            'detected': bool(matches),
            'value_if_parseable': value,
            'evidence_lines': matches[-5:],
        }

    resolved_startup_evidence = {
        'schema_version': 1,
        'source_log': str(server_log_path),
        'source_log_sha256': hashlib.sha256(log_text.encode('utf-8')).hexdigest(),
        'resolved_log_artifact': str(resolved_log_path),
        'evidence_mode': evidence_mode,
        'log_line_count': len(lines),
        'fields': fields,
        'unresolved_fields': [name for name, item in fields.items() if not item['detected']],
        'requested': {
            'max_model_len': 8192,
            'gpu_memory_utilization': 0.97,
            'prefix_caching': True,
            'speculative_draft_enabled': speculative_draft_enabled,
        },
    }
    (ACTIVE_RUN_DIR / 'startup_resolved_config.json').write_text(
        json.dumps(resolved_startup_evidence, indent=2), encoding='utf-8'
    )

last_health_error = None
for attempt in range(90):
    if server_process.poll() is not None:
        capture_resolved_config()
        raise RuntimeError(server_log_path.read_text(encoding='utf-8', errors='replace')[-6000:])
    try:
        with urllib.request.urlopen(f'{BASE_URL}/health', timeout=3) as response:
            if response.status == 200:
                break
    except (urllib.error.URLError, TimeoutError) as error:
        last_health_error = repr(error)
    time.sleep(2)
else:
    capture_resolved_config()
    raise RuntimeError(f'Server did not become healthy: {last_health_error}')

with urllib.request.urlopen(f'{BASE_URL}/v1/models', timeout=10) as response:
    models_payload = json.load(response)
served_models = [entry.get('id') for entry in models_payload.get('data', [])]
if 'LFM2.5-1.2B-Instruct' not in served_models:
    raise RuntimeError(f'Unexpected served model list: {served_models}')

health_artifact = {
    'health_url': f'{BASE_URL}/health',
    'health_status': 200,
    'served_models': served_models,
    'server_pid': server_process.pid,
}
(ACTIVE_RUN_DIR / 'health.json').write_text(json.dumps(health_artifact, indent=2), encoding='utf-8')
capture_resolved_config()
print('Server healthy. Resolved-config excerpts:', ACTIVE_RUN_DIR / 'startup_resolved_config.log')


## 4. Run the repository workload and quick accuracy check

The benchmark is the repository implementation—no reduced Colab copy. It performs the 70-conversation × six-turn workload (420 requests), saves raw `/metrics` immediately before and after it, and fails its process if any request is unsuccessful or has the wrong output-token count. It sends no greedy-comparison request before this workload. A speculative profile additionally requires the benchmark JSON to show measured, run-scoped acceptance evidence. T4 ERS and latency remain stability artifacts only.

In [ ]:
if server_process.poll() is not None:
    raise RuntimeError('The vLLM server is no longer running; rerun the server cell.')

RUN_FULL_WORKLOAD = True
RUN_QUICK_ACCURACY = True
trace_path = REPO_DIR / '019e649f-4e27-74db-82da-920f57b13786' / 'grading-workload-spec.json'
benchmark_output = ACTIVE_RUN_DIR / 'ers-420.json'
WORKLOAD_IDENTITY = {
    'trace_artifact': str(trace_path),
    'trace_sha256': hashlib.sha256(trace_path.read_bytes()).hexdigest(),
    'seed': 42,
    'request_rate': 'inf',
    'expected_requests': 420,
    'output_tokens': 300,
}

if speculative_draft_enabled and not RUN_FULL_WORKLOAD:
    raise RuntimeError(
        'A speculative profile requires the complete 420-request workload; '
        'otherwise run-scoped acceptance metrics cannot be validated.'
    )
if COLAB_PROFILE in {'t4-fp16', 'v6-fp8-smoke'} and not RUN_FULL_WORKLOAD:
    raise RuntimeError('A reusable greedy parent must be captured after the complete 420-request workload.')

def capture_raw_metrics(stage: str) -> tuple[Path, str]:
    metric_path = ACTIVE_RUN_DIR / f'vllm.metrics.{stage}'
    try:
        with urllib.request.urlopen(f'{BASE_URL}/metrics', timeout=15) as response:
            raw_metrics = response.read().decode('utf-8', errors='replace')
    except Exception as error:
        raw_metrics = f'# metrics unavailable during {stage}: {error!r}\n'
    metric_path.write_text(raw_metrics, encoding='utf-8')
    return metric_path, raw_metrics

# This endpoint read is intentionally the only pre-workload operation: no greedy
# generation or prefix prewarming is allowed before the benchmark begins.
metrics_before_path, raw_metrics_before = capture_raw_metrics('before-workload')
benchmark_result = None

if RUN_FULL_WORKLOAD:
    print('Running 420 requests. T4 latency is not an H200 performance signal.')
    benchmark_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'benchmark_ers.py'),
        '--base-url', BASE_URL,
        '--trace', str(trace_path),
        '--tokenizer-path', str(MODEL_DIR),
        '--request-rate', 'inf',
        '--seed', '42',
        '--runs', '1',
        '--output', str(benchmark_output),
    ]
    with (ACTIVE_RUN_DIR / 'benchmark.log').open('w', encoding='utf-8') as benchmark_log:
        benchmark_result = subprocess.run(
            benchmark_command, stdout=benchmark_log, stderr=subprocess.STDOUT, text=True, check=False
        )
metrics_after_path, raw_metrics_after = capture_raw_metrics('after-workload')
# Keep the old filename as a convenience alias for tools that only expect the final snapshot.
(ACTIVE_RUN_DIR / 'vllm.metrics').write_text(raw_metrics_after, encoding='utf-8')

if benchmark_result is not None and benchmark_result.returncode != 0:
    raise RuntimeError(
        'The 420-request workload failed; inspect ' + str(ACTIVE_RUN_DIR / 'benchmark.log')
    )

# Bind the recorder-facing summary to the raw per-request records. A success-rate
# aggregate alone is insufficient: every one of the 420 completions must be 300 tokens.
raw_workload_evidence_artifact = ACTIVE_RUN_DIR / 'raw_workload_evidence.json'
raw_workload_evidence = {
    'schema_version': 1,
    'required': RUN_FULL_WORKLOAD,
    'raw_benchmark_artifact': str(benchmark_output),
    'raw_benchmark_sha256': hashlib.sha256(benchmark_output.read_bytes()).hexdigest() if benchmark_output.is_file() else None,
    'workload': WORKLOAD_IDENTITY,
    'passed': None,
}
workload_evidence_errors = []
if RUN_FULL_WORKLOAD:
    try:
        benchmark_report = json.loads(benchmark_output.read_text(encoding='utf-8'))
    except Exception as error:
        workload_evidence_errors.append(f'could not parse raw benchmark JSON: {error!r}')
        benchmark_report = None
    if isinstance(benchmark_report, dict):
        benchmark_runs = benchmark_report.get('runs')
        if not isinstance(benchmark_runs, list) or len(benchmark_runs) != 1:
            workload_evidence_errors.append('raw benchmark must contain exactly one run')
        else:
            benchmark_run = benchmark_runs[0]
            if not isinstance(benchmark_run, dict):
                workload_evidence_errors.append('raw benchmark run is not an object')
                benchmark_run = {}
            raw_requests = benchmark_run.get('requests')
            if not isinstance(raw_requests, list):
                workload_evidence_errors.append('raw benchmark run has no request records')
                raw_requests = []
            expected_requests = WORKLOAD_IDENTITY['expected_requests']
            expected_tokens = WORKLOAD_IDENTITY['output_tokens']
            if benchmark_run.get('expected_requests') != expected_requests:
                workload_evidence_errors.append('summary expected_requests does not equal 420')
            if benchmark_run.get('observed_requests') != expected_requests or len(raw_requests) != expected_requests:
                workload_evidence_errors.append('raw benchmark does not contain exactly 420 requests')
            if benchmark_run.get('successful_requests') != expected_requests:
                workload_evidence_errors.append('summary successful_requests does not equal 420')
            if benchmark_run.get('failed_requests') != 0:
                workload_evidence_errors.append('summary failed_requests is not zero')
            per_request = []
            for index, request in enumerate(raw_requests, start=1):
                if not isinstance(request, dict):
                    workload_evidence_errors.append(f'request {index} is not an object')
                    continue
                output_tokens = request.get('output_tokens')
                success = request.get('success') is True
                if not success or output_tokens != expected_tokens:
                    workload_evidence_errors.append(
                        f'request {index} is not a successful exact-{expected_tokens}-token completion'
                    )
                per_request.append({
                    'conversation_id': request.get('conversation_id'),
                    'turn': request.get('turn'),
                    'success': success,
                    'output_tokens': output_tokens,
                })
            raw_workload_evidence.update({
                'expected_requests': expected_requests,
                'observed_requests': benchmark_run.get('observed_requests'),
                'successful_requests': benchmark_run.get('successful_requests'),
                'failed_requests': benchmark_run.get('failed_requests'),
                'request_records_sha256': hashlib.sha256(
                    json.dumps(raw_requests, sort_keys=True, separators=(',', ':')).encode('utf-8')
                ).hexdigest(),
                'per_request_completion_evidence': per_request,
            })
    elif benchmark_report is not None:
        workload_evidence_errors.append('raw benchmark JSON root is not an object')
    raw_workload_evidence['passed'] = not workload_evidence_errors
    raw_workload_evidence['errors'] = workload_evidence_errors
raw_workload_evidence_artifact.write_text(
    json.dumps(raw_workload_evidence, indent=2), encoding='utf-8'
)
if workload_evidence_errors:
    raise RuntimeError(
        'Raw workload evidence failed; inspect raw_workload_evidence.json: ' + '; '.join(workload_evidence_errors[:5])
    )

capture_resolved_config()

speculative_benchmark_gate = {
    'required': speculative_draft_enabled,
    'passed': None,
    'metrics_before_artifact': str(metrics_before_path),
    'metrics_after_artifact': str(metrics_after_path),
}
if speculative_draft_enabled:
    gate_errors = []
    speculative_metrics = None
    drafts_observed = None
    mean_acceptance_length = None
    try:
        benchmark_report = json.loads(benchmark_output.read_text(encoding='utf-8'))
    except Exception as error:
        gate_errors.append(f'could not read benchmark JSON: {error!r}')
        benchmark_report = None
    if benchmark_report is not None:
        benchmark_runs = benchmark_report.get('runs')
        if not isinstance(benchmark_runs, list) or len(benchmark_runs) != 1:
            gate_errors.append('benchmark JSON must contain exactly one completed run')
        else:
            speculative_metrics = benchmark_runs[0].get('speculative_decoding')
            if not isinstance(speculative_metrics, dict):
                gate_errors.append('benchmark JSON has no speculative_decoding result')
            else:
                if speculative_metrics.get('available') is not True:
                    gate_errors.append('speculative metrics are not available')
                if speculative_metrics.get('counter_scope') != 'benchmark_delta':
                    gate_errors.append('speculative metrics are not a benchmark_delta')
                if speculative_metrics.get('acceptance_status') != 'measured':
                    gate_errors.append('speculative acceptance was not measured')
                if speculative_metrics.get('counter_reset_detected') is not False:
                    gate_errors.append('speculative counter reset was detected or not reported')
                counters = speculative_metrics.get('counters')
                if not isinstance(counters, dict):
                    gate_errors.append('speculative counters are missing')
                else:
                    try:
                        drafts_observed = float(counters.get('num_drafts'))
                    except (TypeError, ValueError):
                        gate_errors.append('speculative num_drafts is missing or invalid')
                    else:
                        if drafts_observed <= 0:
                            gate_errors.append('no speculative drafts were observed during the workload')
                try:
                    mean_acceptance_length = float(
                        speculative_metrics.get('mean_acceptance_length')
                    )
                except (TypeError, ValueError):
                    gate_errors.append('mean speculative acceptance length is missing or invalid')
                else:
                    if mean_acceptance_length < 3.5:
                        gate_errors.append(
                            'mean speculative acceptance length is below the required 3.5'
                        )
    speculative_benchmark_gate.update({
        'passed': not gate_errors,
        'minimum_mean_acceptance_length': 3.5,
        'drafts_observed': drafts_observed,
        'mean_acceptance_length': mean_acceptance_length,
        'speculative_decoding': speculative_metrics,
        'errors': gate_errors,
    })
    (ACTIVE_RUN_DIR / 'speculative_benchmark_gate.json').write_text(
        json.dumps(speculative_benchmark_gate, indent=2), encoding='utf-8'
    )
    if gate_errors:
        raise RuntimeError(
            'Speculative benchmark gate failed; inspect speculative_benchmark_gate.json: '
            + '; '.join(gate_errors)
        )

# Greedy requests are deliberately post-workload. Capture a parent in a baseline
# run first; then restart the session in the matching speculative profile so its
# candidate checkout and digest evidence remain immutable.
GREEDY_PARENT_PROFILE = {
    'speculative-draft-smoke': 't4-fp16',
    'speculative-draft-v6-fp8-smoke': 'v6-fp8-smoke',
}
GREEDY_PARENT_CAPTURE_PROFILES = {'t4-fp16', 'v6-fp8-smoke'}
greedy_validation = {
    'required': COLAB_PROFILE in GREEDY_PARENT_CAPTURE_PROFILES or speculative_draft_enabled,
    'stage': 'post-complete-workload',
    'passed': None,
    'artifact': None,
}
if COLAB_PROFILE in GREEDY_PARENT_CAPTURE_PROFILES:
    greedy_artifact = ACTIVE_RUN_DIR / 'greedy-parent-capture.json'
    greedy_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'compare_greedy.py'),
        '--base-url', BASE_URL, '--output', str(greedy_artifact),
    ]
    with (ACTIVE_RUN_DIR / 'greedy-parent-capture.log').open('w', encoding='utf-8') as greedy_log:
        greedy_result = subprocess.run(
            greedy_command, stdout=greedy_log, stderr=subprocess.STDOUT, text=True, check=False
        )
    if greedy_result.returncode != 0 or not greedy_artifact.is_file():
        raise RuntimeError('Baseline greedy capture failed; inspect greedy-parent-capture.log')
    greedy_metadata = {
        'schema_version': 1,
        'captured_after_complete_workload': True,
        'profile': COLAB_PROFILE,
        'repository_commit': REPO_SHA,
        'source_compose_sha256': V6_SOURCE_COMPOSE_SHA256,
        'model_revision_resolved': MODEL_RESOLVED_REVISION,
        'workload_identity': WORKLOAD_IDENTITY,
        'artifact_sha256': hashlib.sha256(greedy_artifact.read_bytes()).hexdigest(),
    }
    greedy_metadata_artifact = ACTIVE_RUN_DIR / 'greedy-parent-capture.metadata.json'
    greedy_metadata_artifact.write_text(json.dumps(greedy_metadata, indent=2), encoding='utf-8')
    reusable_greedy_artifact = GREEDY_BASELINE_ROOT / f'{COLAB_PROFILE}.json'
    reusable_greedy_metadata = GREEDY_BASELINE_ROOT / f'{COLAB_PROFILE}.metadata.json'
    shutil.copy2(greedy_artifact, reusable_greedy_artifact)
    shutil.copy2(greedy_metadata_artifact, reusable_greedy_metadata)
    greedy_validation.update({
        'passed': True,
        'mode': 'parent-capture',
        'artifact': str(greedy_artifact),
        'artifact_sha256': greedy_metadata['artifact_sha256'],
        'metadata_artifact': str(greedy_metadata_artifact),
        'reusable_artifact': str(reusable_greedy_artifact),
        'reusable_metadata': str(reusable_greedy_metadata),
    })
elif speculative_draft_enabled:
    expected_parent_profile = GREEDY_PARENT_PROFILE[COLAB_PROFILE]
    default_parent_artifact = GREEDY_BASELINE_ROOT / f'{expected_parent_profile}.json'
    expected_parent_artifact = Path(
        os.environ.get('VIETTEL_GREEDY_PARENT_ARTIFACT', str(default_parent_artifact))
    ).expanduser()
    default_parent_metadata = expected_parent_artifact.with_suffix('.metadata.json')
    expected_parent_metadata = Path(
        os.environ.get('VIETTEL_GREEDY_PARENT_METADATA', str(default_parent_metadata))
    ).expanduser()
    parent_errors = []
    if not expected_parent_artifact.is_file():
        parent_errors.append('missing greedy parent artifact: ' + str(expected_parent_artifact))
    if not expected_parent_metadata.is_file():
        parent_errors.append('missing greedy parent metadata: ' + str(expected_parent_metadata))
    parent_metadata = None
    if not parent_errors:
        try:
            parent_metadata = json.loads(expected_parent_metadata.read_text(encoding='utf-8'))
        except Exception as error:
            parent_errors.append(f'could not parse greedy parent metadata: {error!r}')
    if not parent_errors and not isinstance(parent_metadata, dict):
        parent_errors.append('greedy parent metadata must be a JSON object')
    if isinstance(parent_metadata, dict):
        expected_metadata = {
            'captured_after_complete_workload': True,
            'profile': expected_parent_profile,
            'source_compose_sha256': V6_SOURCE_COMPOSE_SHA256,
            'model_revision_resolved': MODEL_RESOLVED_REVISION,
        }
        for key, expected_value in expected_metadata.items():
            if parent_metadata.get(key) != expected_value:
                parent_errors.append(f'greedy parent metadata {key} mismatch')
        if parent_metadata.get('workload_identity') != WORKLOAD_IDENTITY:
            parent_errors.append('greedy parent workload identity mismatch')
        if parent_metadata.get('artifact_sha256') != hashlib.sha256(expected_parent_artifact.read_bytes()).hexdigest():
            parent_errors.append('greedy parent artifact SHA-256 mismatch')
    if parent_errors:
        greedy_validation.update({
            'passed': False, 'mode': 'speculative-expected-parent',
            'expected_parent_profile': expected_parent_profile,
            'expected_parent_artifact': str(expected_parent_artifact),
            'expected_parent_metadata': str(expected_parent_metadata),
            'errors': parent_errors,
        })
        (ACTIVE_RUN_DIR / 'greedy_validation.json').write_text(
            json.dumps(greedy_validation, indent=2), encoding='utf-8'
        )
        raise RuntimeError('Speculative greedy comparison requires a compatible post-workload parent: ' + '; '.join(parent_errors))
    expected_copy = ACTIVE_RUN_DIR / 'greedy-parent-expected.json'
    expected_metadata_copy = ACTIVE_RUN_DIR / 'greedy-parent-expected.metadata.json'
    shutil.copy2(expected_parent_artifact, expected_copy)
    shutil.copy2(expected_parent_metadata, expected_metadata_copy)
    greedy_artifact = ACTIVE_RUN_DIR / 'greedy-speculative-comparison.json'
    greedy_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'compare_greedy.py'),
        '--base-url', BASE_URL, '--expected', str(expected_copy), '--output', str(greedy_artifact),
    ]
    with (ACTIVE_RUN_DIR / 'greedy-speculative-comparison.log').open('w', encoding='utf-8') as greedy_log:
        greedy_result = subprocess.run(
            greedy_command, stdout=greedy_log, stderr=subprocess.STDOUT, text=True, check=False
        )
    comparison = None
    if greedy_artifact.is_file():
        try:
            comparison = json.loads(greedy_artifact.read_text(encoding='utf-8'))
        except Exception:
            comparison = None
    matches_expected = bool(isinstance(comparison, dict) and comparison.get('matches_expected') is True)
    greedy_validation.update({
        'passed': greedy_result.returncode == 0 and matches_expected,
        'mode': 'speculative-expected-parent',
        'expected_parent_profile': expected_parent_profile,
        'expected_parent_artifact': str(expected_copy),
        'expected_parent_metadata': str(expected_metadata_copy),
        'artifact': str(greedy_artifact),
        'artifact_sha256': hashlib.sha256(greedy_artifact.read_bytes()).hexdigest() if greedy_artifact.is_file() else None,
        'matches_expected': matches_expected,
    })
    if not greedy_validation['passed']:
        (ACTIVE_RUN_DIR / 'greedy_validation.json').write_text(
            json.dumps(greedy_validation, indent=2), encoding='utf-8'
        )
        raise RuntimeError('Speculative greedy comparison failed; inspect greedy-speculative-comparison.json and its log.')

(ACTIVE_RUN_DIR / 'greedy_validation.json').write_text(
    json.dumps(greedy_validation, indent=2), encoding='utf-8'
)

if RUN_QUICK_ACCURACY:
    quick_accuracy_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'test_accuracy.py'),
        '--base-url', BASE_URL, '--mode', 'quick',
        '--quick-output', str(ACTIVE_RUN_DIR / 'quick_accuracy.json'),
    ]
    with (ACTIVE_RUN_DIR / 'quick_accuracy.log').open('w', encoding='utf-8') as accuracy_log:
        quick_accuracy_result = subprocess.run(
            quick_accuracy_command, stdout=accuracy_log, stderr=subprocess.STDOUT, text=True, check=False
        )
    if quick_accuracy_result.returncode != 0:
        raise RuntimeError('Quick accuracy check failed; inspect quick_accuracy.log')

print('Workload and quick-accuracy artifacts:', ACTIVE_RUN_DIR)


## 5. Full GPQA gate and artifact download

Full GPQA is mandatory for either speculative profile and defaults to enabled there; retain the opt-in switch only for ordinary baseline smoke tests. The final cell builds a zip containing the repository SHA, vLLM/Torch/CUDA/GPU preflight, server command, startup-resolved configuration, logs, health result, workload JSON, raw before/after metrics, speculative gate evidence when applicable, and GPQA output.

In [ ]:
# A draft candidate cannot become a portal candidate without full GPQA. Baseline
# runs may opt in with VIETTEL_RUN_FULL_GPQA=1; speculative runs may not disable it.
def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    normalized = raw.strip().lower()
    if normalized in {'1', 'true', 'yes', 'on'}:
        return True
    if normalized in {'0', 'false', 'no', 'off'}:
        return False
    raise ValueError(f'{name} must be one of 1/0, true/false, yes/no, on/off')

requested_full_gpqa = env_bool('VIETTEL_RUN_FULL_GPQA', False)
if speculative_draft_enabled and 'VIETTEL_RUN_FULL_GPQA' in os.environ and not requested_full_gpqa:
    raise RuntimeError('A speculative profile requires full GPQA; VIETTEL_RUN_FULL_GPQA cannot disable it.')
RUN_FULL_GPQA = speculative_draft_enabled or requested_full_gpqa
GPQA_OUTPUT_DIR = ACTIVE_RUN_DIR / 'gpqa_diamond'
GPQA_RESULT_JSON = None
gpqa_accuracy = None
gpqa_result = None

if RUN_FULL_GPQA:
    run_checked([
        sys.executable, '-m', 'uv', 'pip', 'install', '--system',
        '--torch-backend=cu129',
        '--index-strategy', 'unsafe-best-match', 'lm-eval[api]>=0.4.9',
    ])
    gpqa_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'test_accuracy.py'),
        '--base-url', BASE_URL, '--mode', 'gpqa', '--task', 'gpqa_diamond',
        '--concurrency', '4', '--output', str(GPQA_OUTPUT_DIR),
    ]
    with (ACTIVE_RUN_DIR / 'gpqa.log').open('w', encoding='utf-8') as gpqa_log:
        gpqa_result = subprocess.run(
            gpqa_command, stdout=gpqa_log, stderr=subprocess.STDOUT, text=True, check=False
        )
    if gpqa_result.returncode != 0:
        raise RuntimeError('Full GPQA failed; inspect gpqa.log before considering a submission.')
    gpqa_candidates = []
    for candidate_path in sorted(GPQA_OUTPUT_DIR.rglob('*.json')):
        try:
            candidate_payload = json.loads(candidate_path.read_text(encoding='utf-8'))
        except Exception:
            continue
        results = candidate_payload.get('results') if isinstance(candidate_payload, dict) else None
        if not isinstance(results, dict):
            continue
        for task_name, task_metrics in results.items():
            if 'gpqa' not in str(task_name).lower() or not isinstance(task_metrics, dict):
                continue
            accuracy_value = next((
                value for key, value in task_metrics.items()
                if str(key).startswith('acc,') and not str(key).endswith('_stderr')
            ), None)
            try:
                accuracy = float(accuracy_value)
            except (TypeError, ValueError):
                continue
            if 0.0 <= accuracy <= 1.0:
                gpqa_candidates.append((candidate_path, candidate_payload, accuracy))
    if len(gpqa_candidates) != 1:
        raise RuntimeError(
            'Full GPQA did not produce exactly one parseable GPQA result JSON; inspect gpqa.log and ' + str(GPQA_OUTPUT_DIR)
        )
    _, gpqa_payload, gpqa_accuracy = gpqa_candidates[0]
    GPQA_RESULT_JSON = ACTIVE_RUN_DIR / 'gpqa_diamond.results.json'
    GPQA_RESULT_JSON.write_text(json.dumps(gpqa_payload, indent=2), encoding='utf-8')
    if speculative_draft_enabled and gpqa_accuracy < 0.32:
        raise RuntimeError(f'Full GPQA accuracy {gpqa_accuracy:.4f} is below the required 0.32.')
else:
    print('Full GPQA is skipped for this non-speculative smoke run. Set VIETTEL_RUN_FULL_GPQA=1 before setup to run it.')

capture_resolved_config()
server_log_handle.flush()

def artifact_sha256(path: Path) -> dict | None:
    if not path.exists():
        return None
    if path.is_file():
        return {'path': str(path), 'kind': 'file', 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    digest = hashlib.sha256()
    file_count = 0
    for child in sorted((item for item in path.rglob('*') if item.is_file()), key=lambda item: str(item.relative_to(path))):
        relative = str(child.relative_to(path)).replace('\\', '/')
        digest.update(relative.encode('utf-8'))
        digest.update(b'\0')
        digest.update(hashlib.sha256(child.read_bytes()).digest())
        file_count += 1
    return {'path': str(path), 'kind': 'directory', 'sha256': digest.hexdigest(), 'file_count': file_count}

greedy_artifact_path = Path(greedy_validation['artifact']) if greedy_validation.get('artifact') else None
artifact_fingerprints = {
    'server_config': artifact_sha256(ACTIVE_RUN_DIR / 'server_config.json'),
    'source_equivalent_command': artifact_sha256(source_equivalent_command_artifact),
    'benchmark_output': artifact_sha256(benchmark_output),
    'raw_workload_evidence': artifact_sha256(raw_workload_evidence_artifact),
    'gpqa_results': artifact_sha256(GPQA_RESULT_JSON) if GPQA_RESULT_JSON else None,
    'startup_resolved_log': artifact_sha256(ACTIVE_RUN_DIR / 'startup_resolved_config.log'),
    'startup_resolved_evidence': artifact_sha256(ACTIVE_RUN_DIR / 'startup_resolved_config.json'),
    'greedy_artifact': artifact_sha256(greedy_artifact_path) if greedy_artifact_path else None,
}
artifact_hashes = {
    # These scalar names are consumed directly by record_submission.py.
    'metrics': artifact_fingerprints['benchmark_output']['sha256'],
    'raw_workload_evidence': artifact_fingerprints['raw_workload_evidence']['sha256'],
    'source_equivalent_command': artifact_fingerprints['source_equivalent_command']['sha256'],
    'gpqa': artifact_fingerprints['gpqa_results']['sha256'] if artifact_fingerprints['gpqa_results'] else None,
    'greedy_comparison': artifact_fingerprints['greedy_artifact']['sha256'] if artifact_fingerprints['greedy_artifact'] else None,
    'resolved_vllm_config': artifact_fingerprints['startup_resolved_evidence']['sha256'],
    'startup_log': artifact_sha256(server_log_path)['sha256'],
}
run_manifest = {
    'repository_commit': REPO_SHA,
    'repository_ref_requested': REPO_REF,
    'profile': COLAB_PROFILE,
    'server_pid': server_process.pid if server_process.poll() is None else None,
    'server_returncode': server_process.poll(),
    'artifact_directory': str(ACTIVE_RUN_DIR),
    'server_config_artifact': str(ACTIVE_RUN_DIR / 'server_config.json'),
    'source_equivalent_command_artifact': str(source_equivalent_command_artifact),
    'offline_serving': offline_serving_env,
    'metrics_artifact': str(ACTIVE_RUN_DIR / 'vllm.metrics'),
    'benchmark_output_artifact': str(benchmark_output),
    'raw_workload_evidence_artifact': str(raw_workload_evidence_artifact),
    'gpqa_result_artifact': str(GPQA_RESULT_JSON) if GPQA_RESULT_JSON else None,
    'greedy_comparison_artifact': str(greedy_artifact_path) if greedy_artifact_path else None,
    'resolved_vllm_config_artifact': str(ACTIVE_RUN_DIR / 'startup_resolved_config.json'),
    'startup_log_artifact': str(server_log_path),
    'metrics_before_artifact': str(metrics_before_path),
    'metrics_after_artifact': str(metrics_after_path),
    'workload_identity': WORKLOAD_IDENTITY,
    'workload': WORKLOAD_IDENTITY,
    'raw_workload_evidence': raw_workload_evidence,
    'portal_candidate': PORTAL_CANDIDATE,
    'source_equivalent_preflight': bool(
        PORTAL_CANDIDATE and PORTAL_CANDIDATE['source_equivalent_preflight']
    ),
    'startup_resolved_evidence_artifact': str(ACTIVE_RUN_DIR / 'startup_resolved_config.json'),
    'greedy_validation': greedy_validation,
    'full_gpqa': {
        'required': speculative_draft_enabled,
        'requested_for_baseline': requested_full_gpqa,
        'ran': RUN_FULL_GPQA,
        'output_directory': str(GPQA_OUTPUT_DIR) if RUN_FULL_GPQA else None,
        'artifact': str(GPQA_RESULT_JSON) if GPQA_RESULT_JSON else None,
        'accuracy': gpqa_accuracy,
        'returncode': gpqa_result.returncode if gpqa_result is not None else None,
    },
    'artifact_sha256': artifact_hashes,
    'artifact_fingerprints': artifact_fingerprints,
    'speculative_draft': ({
        'enabled': True,
        'config': speculative_config,
        'tokenizer_compatibility_artifact': str(RUN_DIR / 'tokenizer_compatibility.json'),
        'tokenizer_compatible': bool(tokenizer_validation and tokenizer_validation.get('compatible')),
        'benchmark_gate_artifact': str(ACTIVE_RUN_DIR / 'speculative_benchmark_gate.json'),
        'benchmark_gate': speculative_benchmark_gate,
    } if speculative_draft_enabled else {'enabled': False}),
    'note': 'Colab T4 artifacts validate functionality and accuracy only; do not infer H200 latency.',
}
(ACTIVE_RUN_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')
# Archive RUN_DIR rather than only ACTIVE_RUN_DIR so the preflight environment,
# package list, and GPU probe from the setup cell accompany this server run.
archive_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print(f'Artifact archive: {archive_path}')

DOWNLOAD_ARTIFACTS = True
if DOWNLOAD_ARTIFACTS:
    try:
        from google.colab import files
        files.download(archive_path)
    except ImportError:
        print('Not running in Colab; download the archive from the path above.')
